# Categorize Responses in the Behavioural Set

In [ ]:
import sys, os
from pathlib import Path 

PROJECT_ROOT = Path("/Users/robertagarcia/Desktop/learning/bert_symptom_ner")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

sys.path

import torch
import json
from typing import List, Literal, Optional, Union, Dict
from enum import Enum
from collections import Counter
import datetime
from dataclasses import dataclass
from pydantic import BaseModel, Field
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Local imports
from config import settings
from gcp_utils import download_from_gcs
from inference.v01.inference_utils import predict_word_level, word_labels_to_spans
from error_analysis.error_categorization import ErrorCategorizer
from error_analysis.error_taxonomy import BehaviouralExample

VERSION = os.environ["VERSION"]
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1" 
RUN_IDX = 2
INFERENCE_PIPELINE_VERSION = "v01"

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import json

LOCAL_MODEL_DIR = f"{PROJECT_ROOT}/{VERSION}/downloaded_models/dmis-lab/biobert-base-cased-v1.1/run_{RUN_IDX}"

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)
model = AutoModelForTokenClassification.from_pretrained(LOCAL_MODEL_DIR)
print(f"✅ Loaded model and tokenizer from {LOCAL_MODEL_DIR}")


In [ ]:

# 2) Load label mappings from the local JSON (same labels across all versions)
with open("../v01/data/id2label.json", "r") as f:
    id2label = {int(k): v for k, v in json.load(f).items()}

label2id = {v: k for k, v in id2label.items()}

print(f"✅ Loaded {len(id2label)} labels: {id2label}")

In [ ]:
# Move model to device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")
model.to(device)
model.eval()
print("✅ Model is ready to be used")

# Load the Behavioural Set

In [ ]:
# Load behavioural evaluation set
print("\n📂 Loading behavioural evaluation set...")
with open(f"{PROJECT_ROOT}/v01/behavioural_set.json", "r") as f:
    behavioural_set = json.load(f)


# Fit the behavioural set into the dataclasses
for k,examples in behavioural_set.items():
    for i,ex in enumerate(examples):
        behavioural_set[k][i] = BehaviouralExample(**ex)



# Instantiate the Error Categorizer

In [ ]:
error_categorizer = ErrorCategorizer()

**Run on a single example**

In [ ]:
print("Working example:")
my_example = behavioural_set['Long, realistic clinical sentences (THIS IS GOLD 🥇)'][0]
text = my_example.example
print(f"\t{text}")
tokens, token_labels, word_ids, words, word_labels, word_offsets = predict_word_level(
        text=text,
        model=model,
        tokenizer=tokenizer,
        id2label=id2label,
        device=device,
    )
spans = word_labels_to_spans(text=text, word_offsets=word_offsets, word_labels=word_labels)
print("SPANS:")
for s in spans:
    print(f"\t{s}")

# Example to show how error_categorizer._is_entity_in_spans(a['ent'], spans) works
sample_entity_data = my_example.entities_with_labels[0]
ent = sample_entity_data['ent']
idx = error_categorizer._is_entity_in_spans(ent, spans)
print(f"The entity {ent} was found in the following span: {spans[idx]}")
# See which entities from the behavioural examples are in:
# For present entities, we also track which span index they were found in
present_with_indices = [(e, error_categorizer._is_entity_in_spans(e["ent"], spans)) for e in my_example.entities_with_labels]
present = [(e,idx) for e, idx in present_with_indices if idx is not None]
missing = [(e,idx) for e, idx in present_with_indices if idx is None]

print(f"present_with_indices:\n\t{present_with_indices}")
print(f"present:\n\t{present}")
print(f"missing:\n\t{missing}")

# Run Error Categorization thoughout the entire dataset

In [ ]:
from error_analysis.error_categorization import _run_through_behavioural_set


_run_through_behavioural_set

In [ ]:
results = _run_through_behavioural_set(
    behavioural_set=behavioural_set,
    error_categorizer=error_categorizer,
    model=model,
    id2label=id2label,
    tokenizer=tokenizer,
    device=device
)

# These results are very important they dictate the next steps for improving the model!
results["error_counts"]